# GigaGraph v9.0: 'NoProp' Denoising Prototype (100M)
### March 2025: BP-Free Sequential Denoising

**Innovation:** This model eliminates the global forward/backward pass. Each block trains independently to 'clean' a noisy representation of the target label, conditioned on the current input context.

In [1]:
import os, sys, torch
import torch.nn.functional as F
from tqdm import tqdm
import wandb

# Source Sync (GitHub -> Kaggle)
REPO_URL = 'https://github.com/ey3lock3r/gnn-llm.git'
if not os.path.exists('.git'): !git init .
!git remote add origin {REPO_URL} || git remote set-url origin {REPO_URL}
!git fetch origin && git reset --hard origin/master

import noprop_gnn, data_pipeline
from noprop_gnn import NoPropGNN
from data_pipeline import GigaDataPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB_SIZE, D_MODEL, DEPTH = 128256, 768, 8
model = NoPropGNN(VOCAB_SIZE, D_MODEL, DEPTH, device=device)
pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=2, seq_len=512)

wandb.init(project='gigagraph-v9-noprop-100m', id='noprop-proto-1')
print(f"🚀 NoProp v9.0 Prototype Launching on {device}...")

In [2]:
# v9.0 'NoProp' Parallel Block Training Loop
pbar = tqdm(loader)
for i, batch in enumerate(pbar):
    x_ids = batch.to(device)
    y_ids = torch.roll(x_ids, -1, dims=1)
    
    # Forward: Get embeddings for input and target
    x_embed = model.embed(x_ids)
    y_embed = model.embed(y_ids)
    
    # 1. Generate Noise Path (Diffusion Path for local targets)
    # Creates targets: [noisy_latent_0, noisy_latent_1, ..., clean_y]
    path = model.generate_noise_path(y_embed, DEPTH)
    
    # 2. Parallel Local Training (No Global Sync)
    total_loss = 0
    for d in range(DEPTH):
        block = model.blocks[d]
        # Block d: learns to move path[d] (noisier) -> path[d+1] (cleaner)
        loss = block.train_block(x_embed, path[d].detach(), path[d+1].detach())
        total_loss += loss
        
    if i % 10 == 0:
        wandb.log({"total_loss": total_loss / DEPTH, "step": i})
        pbar.set_postfix({'loss': f'{total_loss/DEPTH:.4f}'})

wandb.finish()